> **Same contract. Polars, DuckDB, or Spark. Zero code changes.**

# Engine Portability & Scale — One Contract, Three Engines, Zero Lock-In

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/03_engine_scale.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/03_engine_scale.ipynb)

You built your pipeline on Polars because it was fast on your laptop. Now production says Spark. Three months of rewrites.

LakeLogic abstracts the engine. The **same YAML contract** runs on Polars, DuckDB, and Spark — with identical results. This notebook proves it, then layers in **SCD2 merges, incremental watermarks, parallel multi-contract execution, targeted backfills, and Iceberg materialization** — all engine-agnostic.

In [ ]:
# Install lakelogic
!pip install -q lakelogic[polars,duckdb]

import urllib.request
import os

if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll

### ⚙️ Execution Engine

In [ ]:
# Select execution engine (Colab comes with PySpark pre-installed)
ENGINE = "duckdb"  # 'polars' , 'spark'

---
## 1. Engine-Agnostic Proof — Zero Code Changes

**The Problem:** You built your pipeline on Polars/DuckDB. Now it needs to run on Spark in production. Rewriting 2,000 lines of DataFrame logic isn't a weekend project.

**The Solution:** LakeLogic compiles SQL-first rules to each engine's dialect. Same contract, same results.

In [ ]:
import os
import polars as pl

# ── Clean slate: remove stale files from previous runs ────────────
for _f in ["03_engine_scale_demo/engine_test.yaml", "engine_test_source.parquet"]:
    if os.path.exists(_f):
        os.remove(_f)

contract = s.write_contract(
    """
version: 1.0.0
dataset: engine_test

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: email
      type: string
      required: true
    - name: score
      type: integer

quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
    - name: score_range
      sql: "score BETWEEN 0 AND 100"
""",
    "03_engine_scale_demo/engine_test.yaml",
)

# Generate ONE dataset and persist it, so every engine validates the exact
# same 500 rows — otherwise "same results" would be comparing different data.
source_df = ll.DataGenerator(contract).generate(rows=500, invalid_ratio=0.08, output_format="polars")
source_file = "engine_test_source.parquet"
source_df.write_parquet(source_file)
base_df = pl.read_parquet(source_file)

# Run the SAME contract on two genuinely different engines (both installed above).
# The DuckDB adapter registers the Polars frame via Arrow, so both see identical input.
r_polars = ll.DataProcessor(contract, engine="polars").run(base_df)
r_duckdb = ll.DataProcessor(contract, engine="duckdb").run(base_df)

# Spark is optional — Colab ships PySpark, but it's heavy to spin up.
# Flip run_spark = True to include it in the comparison.
run_spark = False
r_spark = None
if run_spark:
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.appName("LakeLogic").getOrCreate()
    spark_df = spark.read.parquet(source_file)
    r_spark = ll.DataProcessor(contract, engine="spark").run(spark_df)

In [ ]:
# The Proof — same contract + same data, run on genuinely different engines
print("Engine Comparison (same contract, same 500 rows)")
print("=" * 50)
print(f"  Polars : good={r_polars.good_count}, bad={r_polars.bad_count}")
print(f"  DuckDB : good={r_duckdb.good_count}, bad={r_duckdb.bad_count}")

counts = [(r_polars.good_count, r_polars.bad_count), (r_duckdb.good_count, r_duckdb.bad_count)]
if run_spark and r_spark is not None:
    print(f"  Spark  : good={r_spark.good_count}, bad={r_spark.bad_count}")
    counts.append((r_spark.good_count, r_spark.bad_count))

match = all(c == counts[0] for c in counts)
print(f"\n  Identical results across engines : {match}")
print("\nSame contract. Same data. Same results. Zero code changes.")

---
## 2. Dimensional Modeling — SCD2, Merge, Overwrite

**The Problem:** Your dimension table needs history tracking. You manually build `MERGE INTO` SQL, manage `effective_from`/`effective_to` dates, and debug `is_current` flags by hand.

**The Solution:** Declare `materialization.strategy: scd2` in the contract — LakeLogic generates all SCD2 columns, applies merge logic, and manages version tracking. Zero SQL required.

In [ ]:
import yaml

# ── Show what a fully-declared dimensional contract looks like ────────
scd2_yaml = """
version: 1.0.0
dataset: dim_customers
info:
  title: gold_dim_customers
  target_layer: gold

primary_key: [customer_id]

model:
  fields:
    - name: customer_id
      type: integer
      required: true
    - name: name
      type: string
    - name: email
      type: string
    - name: tier
      type: string

materialization:
  strategy: scd2
  scd2:
    track_columns: [name, email, tier]
    timestamp_field: updated_at
    surrogate_key: _sk
    effective_from_field: effective_from
    effective_to_field: effective_to
    current_flag_field: is_current
    end_date_default: "9999-12-31"
    version_column: "_version"             # ROW_NUMBER per business key
    change_reason_column: "_change_reason" # "initial_load", "email,tier changes", etc.

    # Unknown member (late-arriving fact fallback)
    unknown_member:
      enabled: true
      surrogate_key_value: "-1"
"""

# Parse & validate the contract
from lakelogic.core.models import DataContract

scd2_contract = ll.DataContract(**yaml.safe_load(scd2_yaml))

print("\u2500" * 60)
print("DIMENSIONAL MODELING: SCD2 Contract")
print("\u2500" * 60)
print(f"  Strategy       : {scd2_contract.materialization.strategy}")
print(f"  Primary key    : {scd2_contract.primary_key}")
print(f"  Track columns  : {scd2_contract.materialization.scd2['track_columns']}")
print(f"  Surrogate key  : {scd2_contract.materialization.scd2['surrogate_key']}")
print(f"  Effective from : {scd2_contract.materialization.scd2['effective_from_field']}")
print(f"  Effective to   : {scd2_contract.materialization.scd2['effective_to_field']}")
print(f"  Current flag   : {scd2_contract.materialization.scd2['current_flag_field']}")
print(f"  version        : {scd2_contract.materialization.scd2['version_column']}")
print(f"  change reason  : {scd2_contract.materialization.scd2['change_reason_column']}")
print(f"  unknown_member : {scd2_contract.materialization.scd2['unknown_member']}")


# ── Show all supported strategies ─────────────────────────────────────
strategies = {
    "append": "Fact tables — new rows added, never updated",
    "merge": "SCD Type 1 — upsert by natural key, latest value wins",
    "scd2": "SCD Type 2 — full history with effective dates",
    "overwrite": "Periodic snapshot — drop & replace on each run",
}

print("\n" + "\u2500" * 60)
print("ALL MATERIALIZATION STRATEGIES")
print("\u2500" * 60)
for strat, desc in strategies.items():
    marker = "\u2716" if strat == scd2_contract.materialization.strategy else " "
    print(f"  [{marker}] {strat:10s} — {desc}")
print("\n\u2705 All declared in YAML. No manual MERGE INTO SQL required.")

### The Proof — Let's see SCD2 in action
We'll generate 3 rows, run them through the processor, and see how LakeLogic automatically injects and populates the tracking columns without any SQL.

In [ ]:
import os
import shutil

# One path for both the write target and the read-back, so they can't drift.
SCD2_OUT = "03_engine_scale_demo/parallel_demo/gold_dim_customers"

# ── Clean slate: remove stale files from previous runs ────────────
for _target in [SCD2_OUT, "03_engine_scale_demo/dim_customers.yaml"]:
    if os.path.isdir(_target):
        shutil.rmtree(_target)
    elif os.path.isfile(_target):
        os.remove(_target)

# Generate some initial data
scd2_path = s.write_contract(scd2_yaml, "03_engine_scale_demo/dim_customers.yaml")
scd2_source = ll.DataGenerator(scd2_path).generate(rows=3, output_format=ENGINE)
print("1. INCOMING SOURCE DATA:\n")
display(s.preview(scd2_source))

# Run the pipeline (LakeLogic automatically handles the SCD2 merge logic)
p_scd2 = ll.DataProcessor(scd2_path, engine=ENGINE)
good_scd2, bad_scd2 = p_scd2.run(scd2_source, materialize=True, materialize_target=SCD2_OUT)

print("\n2. AFTER LAKELOGIC SCD2 PROCESSING (Notice the injected tracking columns):\n")
try:
    materialized = s.read_table(SCD2_OUT)
except Exception:
    materialized = good_scd2

# Show the business + SCD2 tracking columns (hide only internal lineage bookkeeping).
_cols = [c for c in s.to_records(materialized, 1)[0] if not c.startswith("_lakelogic_")]
display(s.preview(materialized, columns=_cols))

---
## 3. Incremental Processing — `pipeline_log` Watermark

**The Problem:** Your nightly job reprocesses 10 million rows even though only 500 changed. Compute costs scale with total volume instead of change volume.

**The Solution:** LakeLogic's `pipeline_log` watermark strategy tracks which files have been processed by their modification time. On the next run, only **new files** are loaded.

In [ ]:
import os
import shutil

# ── Clean slate for demo ─────────────────────────────────────────────
DEMO_DIR = "./incremental_demo"
LANDING = f"{DEMO_DIR}/landing"

if os.path.exists(DEMO_DIR):
    shutil.rmtree(DEMO_DIR)
os.makedirs(LANDING, exist_ok=True)

# ── Contract with source.type = landing, load_mode = incremental ────
inc_contract = s.write_contract(
    """
version: 1.0.0
dataset: orders
info:
  title: bronze_orders
  target_layer: bronze

source:
  type: landing
  path: "./incremental_demo/landing"
  format: ndjson
  load_mode: incremental
  watermark_strategy: pipeline_log

metadata:
  run_log_dir: "./incremental_demo/logs"

model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float
    - name: status
      type: string

quality:
  row_rules:
    - name: positive_amount
      sql: "amount > 0"
""",
    "03_engine_scale_demo/orders_inc.yaml",
)

# ── FILE 1: 50 orders land in the landing zone ───────────────────────
batch1 = ll.DataGenerator(inc_contract).generate(rows=50, output_format=ENGINE)
batch1.write_ndjson(f"{LANDING}/orders_batch_1.json")
print(f"File 1: wrote {s.row_count(batch1)} rows to orders_batch_1.json")

# ── RUN 1: Initial load (no prior watermark) ────────────────────────
proc = ll.DataProcessor(inc_contract, engine=ENGINE)
g1, b1 = proc.run_source()
r1 = proc.last_report

print(f"\n{'=' * 50}")
print("RUN 1 (initial load)")
print(f"{'=' * 50}")
print(f"  Files in landing : {len(os.listdir(LANDING))}")
print(f"  Rows loaded      : {r1.get('counts', {}).get('source', '?')}")
print(f"  Good / Bad       : {r1.get('counts', {}).get('good', '?')} / {r1.get('counts', {}).get('quarantined', '?')}")

In [ ]:
import time

time.sleep(1)  # Ensure mtime of File 2 is strictly after Run 1's watermark

# ── FILE 2: 20 new orders arrive ────────────────────────────────────
batch2 = ll.DataGenerator(inc_contract).generate(rows=20, output_format=ENGINE)
batch2.write_ndjson(f"{LANDING}/orders_batch_2.json")
print(f"File 2: wrote {s.row_count(batch2)} rows to orders_batch_2.json")
print(f"  Landing zone now has: {os.listdir(LANDING)}")

# ── RUN 2: Only new files processed ─────────────────────────────────
g2, b2 = proc.run_source()
r2 = proc.last_report

print(f"\n{'=' * 50}")
print("RUN 2 (incremental)")
print(f"{'=' * 50}")
print(f"  Files in landing : {len(os.listdir(LANDING))} (70 total rows across 2 files)")
print("  Files processed  : 1 (only orders_batch_2.json — batch_1 already processed)")
print(f"  Rows loaded      : {r2.get('counts', {}).get('source', '?')}")
print(f"  Good / Bad       : {r2.get('counts', {}).get('good', '?')} / {r2.get('counts', {}).get('quarantined', '?')}")
print("\npipeline_log watermark: only new files are processed. No reprocessing.")

---
## 4. Parallel Processing — Concurrent Multi-Contract Execution

**The Problem:** You have 8 Bronze contracts with no dependencies between them. Running them sequentially takes 40 minutes.

**The Solution:** `pipeline.run(parallel=True)` groups contracts into dependency **waves** using topological sort. Contracts within the same wave execute concurrently via threads — layer ordering is preserved automatically.

In [ ]:
import os
import shutil
import yaml
from lakelogic.core.registry import DomainRegistry
from lakelogic.pipeline import LakehousePipeline
from IPython.display import HTML, display

# ── Create inline contracts ──────────────────────────────────────────
DAG_DIR = "./parallel_demo"

# ── Clean slate: remove stale files from previous runs ────────────
if os.path.exists(DAG_DIR):
    shutil.rmtree(DAG_DIR)

os.makedirs(f"{DAG_DIR}/contracts/bronze", exist_ok=True)
os.makedirs(f"{DAG_DIR}/contracts/silver", exist_ok=True)

# Bronze: orders (independent)
s.write_contract(
    """
version: 1.0.0
dataset: orders
info:
  title: bronze_orders
  target_layer: bronze
source:
  type: landing
  path: "./parallel_demo/landing/orders"
  format: ndjson
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float
""",
    f"{DAG_DIR}/contracts/bronze/orders.yaml",
)

# Bronze: customers (independent — runs in parallel with orders)
s.write_contract(
    """
version: 1.0.0
dataset: customers
info:
  title: bronze_customers
  target_layer: bronze
source:
  type: landing
  path: "./parallel_demo/landing/customers"
  format: ndjson
model:
  fields:
    - name: customer_id
      type: integer
      required: true
    - name: name
      type: string
    - name: email_address
      type: string
      pii: true
""",
    f"{DAG_DIR}/contracts/bronze/customers.yaml",
)

# Bronze: products (independent — runs in parallel with orders & customers)
s.write_contract(
    """
version: 1.0.0
dataset: products
info:
  title: bronze_products
  target_layer: bronze
source:
  type: landing
  path: "./parallel_demo/landing/products"
  format: ndjson
model:
  fields:
    - name: product_id
      type: integer
      required: true
    - name: name
      type: string
""",
    f"{DAG_DIR}/contracts/bronze/products.yaml",
)

# Silver: customers_enriched
s.write_contract(
    """
version: 1.0.0
dataset: customers_enriched
info:
  title: silver_customers_enriched
  target_layer: silver
source:
  type: table
  path: "./parallel_demo/lakehouse/bronze/bronze_customers"
model:
  fields:
    - name: customer_id
      type: integer
      required: true
    - name: name
      type: string
    - name: email_address
      type: string
      pii: true

""",
    f"{DAG_DIR}/contracts/silver/customers_enriched.yaml",
)

# Silver: orders_enriched (depends on orders + customers → runs AFTER them)
s.write_contract(
    """
version: 1.0.0
dataset: orders_enriched
info:
  title: silver_orders_enriched
  target_layer: silver
source:
  type: table
  path: "./parallel_demo/lakehouse/bronze/bronze_orders"
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float
downstream:
  - type: dashboard
    name: "Weekly Sales Performance"
    platform: power_bi
    url: "https://app.powerbi.com/..."
    owner: "marketing-analytics"

  - type: api
    name: "Order Tracking Service"
    platform: internal
    owner: "backend-team"

""",
    f"{DAG_DIR}/contracts/silver/orders_enriched.yaml",
)

# ── Create _system.yaml with dependency declarations ────────────────
system_yaml = {
    "domain": "demo",
    "system": "ecommerce",
    # ── External sources (for lineage visualization) ──────────────────────────
    "external_sources": [
        {
            "name": "Shopify API",
            "source_domain": "CRM Vendor",
            "catalog_path": "external_storage_path_or_api",
            "consumed_by": ["orders", "customers"],
        },
        {
            "name": "Products System Database",
            "source_domain": "Products Vendor",
            "catalog_path": "external_storage_path_or_api",
            "consumed_by": ["products"],
        },
    ],
    "contracts": [
        {"layer": "bronze", "entity": "orders", "path": "contracts/bronze/orders.yaml", "enabled": True},
        {"layer": "bronze", "entity": "customers", "path": "contracts/bronze/customers.yaml", "enabled": True},
        {"layer": "bronze", "entity": "products", "path": "contracts/bronze/products.yaml", "enabled": True},
        {
            "layer": "silver",
            "entity": "customers_enriched",
            "path": "contracts/silver/customers_enriched.yaml",
            "enabled": True,
        },
        {
            "layer": "silver",
            "entity": "orders_enriched",
            "path": "contracts/silver/orders_enriched.yaml",
            "depends_on": ["customers_enriched"],
            "enabled": True,
        },
    ],
    "environments": {
        "local": {
            "catalog": "local",
            "storage_root": "./parallel_demo/lakehouse",
            "data_root": "./parallel_demo/lakehouse",
            "quarantine_root": "./parallel_demo/lakehouse/_quarantine",
        }
    },
    "storage": {"external_location_root": "./parallel_demo/lakehouse"},
}

sys_path = f"{DAG_DIR}/_system.yaml"
with open(sys_path, "w") as f:
    yaml.dump(system_yaml, f, default_flow_style=False)

# ── Build pipeline ──────────────────────────────────────────────────
registry = DomainRegistry.from_yaml(sys_path, environment="local", storage_mode="direct")
pipeline = LakehousePipeline(registry, engine=ENGINE)

# ── Visualise the DAG — shows parallel waves ────────────────────────
display(HTML(pipeline.visualize_dag()))

# ── Show wave grouping ──────────────────────────────────────────────
from lakelogic.pipeline.runner import LakehousePipeline as _LP

bronze_contracts = [c for c in registry.contracts if c.layer == "bronze"]
waves = _LP._group_by_dependency_level(bronze_contracts)

print("\n" + "\u2500" * 60)
print("PARALLEL EXECUTION PLAN")
print("\u2500" * 60)
print(f"  Bronze layer: {len(bronze_contracts)} contracts")
for i, wave in enumerate(waves):
    entities = [c.entity for c in wave]
    print(f"  Wave {i}: [{', '.join(entities)}] \u2190 {'parallel' if len(entities) > 1 else 'sequential'}")

print("\n  Silver layer: orders_enriched")
print("  \u2514\u2500 depends_on: [orders, customers] \u2192 waits for Bronze to complete")

print("\n\u2705 pipeline.run(parallel=True) executes Wave 0 contracts concurrently.")
print("   Layer ordering (bronze \u2192 silver \u2192 gold) is always preserved.")

---
## 5. Backfill & Reprocessing — Targeted Late-Arriving Data

**The Problem:** A partner sent corrected data for last Tuesday. You need to reload just those records without blowing away the rest of the week.

**The Solution:** `run_source(reprocess_from=..., reprocess_to=...)` lets you surgically reload a date range or specific IDs — the incremental watermark is bypassed for that run only.

In [ ]:
import os
import shutil
import polars as pl  # this section uses Polars as the local analysis engine
from datetime import date, timedelta

# ── Clean slate ─────────────────────────────────────────────────────
BF_DIR = "./backfill_demo"
BF_LANDING = f"{BF_DIR}/landing"
if os.path.exists(BF_DIR):
    shutil.rmtree(BF_DIR)
os.makedirs(BF_LANDING, exist_ok=True)

# ── Contract with source.type = landing + reprocess column ─────────
backfill_contract = s.write_contract(
    """
version: 1.0.0
dataset: daily_events
info:
  title: bronze_daily_events
  target_layer: bronze

source:
  type: landing
  path: "./backfill_demo/landing/*.ndjson"
  format: ndjson

model:
  fields:
    - name: event_id
      type: integer
      required: true
    - name: event_date
      type: string
      required: true
    - name: payload
      type: string

materialization:
  reprocess_date_column: event_date

quality:
  row_rules:
    - name: has_payload
      sql: "payload IS NOT NULL"
""",
    "03_engine_scale_demo/daily_events.yaml",
)

# ── Generate a week of events and write to landing ────────────────
today = date.today()
rows = []
for i in range(7):
    day = (today - timedelta(days=6 - i)).isoformat()
    for j in range(50):
        rows.append({"event_id": i * 50 + j, "event_date": day, "payload": f"data_{i}_{j}"})

full_week = pl.DataFrame(rows)
full_week.write_ndjson(f"{BF_LANDING}/events_full_week.ndjson")
print(f"Full dataset: {len(full_week)} rows across 7 days")
print(full_week.group_by("event_date").len().sort("event_date"))

# ── Full load first ──────────────────────────────────────────────────
bf_proc = ll.DataProcessor(backfill_contract, engine=ENGINE)
g_full, b_full = bf_proc.run_source()
r_full = bf_proc.last_report
print(f"\nFull load: {r_full.get('counts', {}).get('source', '?')} rows")

# ── Targeted backfill: reload just 2 days ──────────────────────────
target_start = (today - timedelta(days=3)).isoformat()
target_end = (today - timedelta(days=2)).isoformat()

g_bp, b_bp = bf_proc.run_source(
    reprocess_from=target_start,
    reprocess_to=target_end,
)
r_bp = bf_proc.last_report

print(f"\n{'=' * 50}")
print(f"BACKFILL: {target_start} to {target_end}")
print(f"{'=' * 50}")
print(f"  Rows reprocessed : {r_bp.get('counts', {}).get('source', '?')}")
print(f"  Good / Bad       : {r_bp.get('counts', {}).get('good', '?')} / {r_bp.get('counts', {}).get('bad', '?')}")
print("\nOnly the targeted date range was reprocessed — rest of the week untouched.")

### 5b. Backfill That Cascades — Reprocessing Downstream Tables

`run_source(...)` above reloads a single source. But the hard part of a backfill is **everything downstream** — the Silver tables and Gold aggregates that derived from those rows. Fix the source, forget one dependent table, and your dashboards quietly stay wrong.

LakeLogic drives the whole cascade from the **dependency graph**. Register `depends_on`, then run the pipeline with a reprocess window (`reprocess_from` / `reprocess_to`). The correction flows through the medallion in dependency order — source first, then every dependent table — and **only the reprocessed window changes downstream**. The rest of history stays untouched.

In [ ]:
from lakelogic import read_delta  # polars' Delta bridge is broken; this routes via Arrow
# ── Downstream cascade: reprocess ONE window, watch it flow through depends_on ──
# Reloading the source is the easy half. The hard half is every dependent table.
# Here: bronze events → silver daily rollup (linked by depends_on). We run a single
# targeted reprocess across the whole pipeline; only the corrected window should
# change downstream — the rest of the week stays untouched.
import os
import shutil
import yaml
import polars as pl
from datetime import date, timedelta
from lakelogic.core.registry import DomainRegistry
from lakelogic.pipeline import LakehousePipeline

CAS = "./cascade_demo"
if os.path.exists(CAS):
    shutil.rmtree(CAS)
for _sub in ["contracts/bronze", "contracts/silver", "landing/events"]:
    os.makedirs(f"{CAS}/{_sub}", exist_ok=True)


def _wc(text, path):
    with open(path, "w") as f:
        f.write(text)


# Bronze: raw events landing — keyed by event_id, dated by event_date (the reprocess column).
_wc(
    f"""
version: 1.0.0
dataset: events
info:
  title: bronze_events
  target_layer: bronze
primary_key: [event_id]
source:
  type: landing
  path: "{CAS}/landing/events/*.ndjson"
  format: ndjson
materialization:
  format: delta
  strategy: merge
  target_path: "{CAS}/lakehouse/bronze/bronze_events"
  reprocess_date_column: event_date
model:
  fields:
    - name: event_id
      type: integer
      required: true
    - name: event_date
      type: string
      required: true
    - name: amount
      type: float
""",
    f"{CAS}/contracts/bronze/events.yaml",
)

# Silver: DEPENDS ON bronze_events (reads its materialized output). Same reprocess column.
_wc(
    f"""
version: 1.0.0
dataset: daily_summary
info:
  title: silver_daily_summary
  target_layer: silver
primary_key: [event_id]
source:
  type: table
  path: "{CAS}/lakehouse/bronze/bronze_events"
materialization:
  format: delta
  strategy: merge
  target_path: "{CAS}/lakehouse/silver/silver_daily_summary"
  reprocess_date_column: event_date
model:
  fields:
    - name: event_id
      type: integer
      required: true
    - name: event_date
      type: string
      required: true
    - name: amount
      type: float
""",
    f"{CAS}/contracts/silver/daily_summary.yaml",
)

with open(f"{CAS}/_system.yaml", "w") as f:
    yaml.dump(
        {
            "domain": "demo",
            "system": "events",
            "contracts": [
                {"layer": "bronze", "entity": "events", "path": "contracts/bronze/events.yaml", "enabled": True},
                {
                    "layer": "silver",
                    "entity": "daily_summary",
                    "path": "contracts/silver/daily_summary.yaml",
                    "depends_on": ["events"],
                    "enabled": True,
                },
            ],
            "environments": {
                "local": {
                    "catalog": "local",
                    "storage_root": f"{CAS}/lakehouse",
                    "data_root": f"{CAS}/lakehouse",
                    "quarantine_root": f"{CAS}/lakehouse/_quarantine",
                }
            },
            "storage": {"external_location_root": f"{CAS}/lakehouse"},
        },
        f,
    )

# One landing file = one source of truth per event. `window` doubles a date range.
today = date.today()


def write_week(window=None, multiplier=2):
    rows = []
    for i in range(7):
        d = (today - timedelta(days=6 - i)).isoformat()
        m = multiplier if (window and window[0] <= d <= window[1]) else 1
        for j in range(50):
            rows.append({"event_id": i * 50 + j, "event_date": d, "amount": float((i * 10 + j) * m)})
    pl.DataFrame(rows).write_ndjson(f"{CAS}/landing/events/events.ndjson")


write_week()  # original week

registry = DomainRegistry.from_yaml(f"{CAS}/_system.yaml", environment="local", storage_mode="direct")
pipeline = LakehousePipeline(registry, engine=ENGINE)

# Show the dependency graph the reprocess will flow through (bronze -> silver).
from IPython.display import HTML, display

display(HTML(pipeline.visualize_dag()))


def silver_by_date():
    df = read_delta(f"{CAS}/lakehouse/silver/silver_daily_summary")
    return df.group_by("event_date").agg(pl.col("amount").sum().round(1).alias("total")).sort("event_date")


# 1) Full run: bronze → silver
pipeline.run(target_layers="bronze,silver")
before = silver_by_date()

# 2) Partner resends CORRECTED data for a 2-day window (amounts doubled for those days).
ts = (today - timedelta(days=3)).isoformat()
te = (today - timedelta(days=2)).isoformat()
write_week(window=(ts, te), multiplier=2)

# 3) ONE targeted reprocess — flows through depends_on: bronze first, then silver.
pipeline.run(target_layers="bronze,silver", reprocess_from=ts, reprocess_to=te)
after = silver_by_date()

# 4) Prove only the reprocessed window changed downstream.
proof = (
    before.join(after, on="event_date", suffix="_after")
    .with_columns((pl.col("total_after") != pl.col("total")).alias("changed_downstream"))
    .sort("event_date")
)

print(f"Reprocess window: {ts} → {te}\n")
print(proof)
print("\n✅ One reprocess window flowed source → dependent table, in depends_on order.")
print("   Only the corrected days changed downstream — the rest of the week is untouched.")

## What You Just Did

Six features that normally require six different engineering projects:

- ✅ **Engine portability** — same contract, Polars + DuckDB + Spark, identical results
- ✅ **Dimensional modeling** — SCD2 / merge / overwrite, declarative
- ✅ **Incremental processing** — `pipeline_log` watermark, no full rescans
- ✅ **Parallel execution** — concurrent multi-contract runs
- ✅ **Targeted backfills** — reprocess a window and cascade it through `depends_on` to dependent tables
- ✅ **Custom Python hooks** — drop in your own logic where YAML can't reach

Vendor lock-in: **zero**.

In [ ]:
import os
import shutil
import polars as pl  # the custom transform below is written in Polars

# ── Clean slate ─────────────────────────────────────────────────────
EXT_DIR = "03_engine_scale_demo/external_logic_demo"
if os.path.exists(EXT_DIR):
    shutil.rmtree(EXT_DIR)
os.makedirs(f"{EXT_DIR}/transforms", exist_ok=True)

# ── Step 1: Write the custom Python transform to disk ────────────
transform_code = """
import polars as pl

def run(df, *, fiscal_year=2026, include_refunds=False, **kwargs):
    \"\"\"Gold-layer aggregation - called by LakeLogic.\"\"\"\n
    # Filter out refunds if requested
    if not include_refunds:
        df = df.filter(pl.col("status") != "refunded")
    return df.group_by("region").agg(
        pl.col("amount").sum().alias("total_revenue"),
        pl.col("order_id").count().alias("order_count"),
    )
"""

transform_path = f"{EXT_DIR}/transforms/revenue_summary.py"
with open(transform_path, "w", encoding="utf-8") as f:
    f.write(transform_code.strip() + "\n")

print("transforms/revenue_summary.py")
print("─" * 60)
print(transform_code.strip())
print("─" * 60)

# ── Step 2: Create the contract referencing the script ────────────
ext_yaml = """
version: 1.0.0
dataset: gold_revenue_summary
info:
  title: gold_revenue_summary
  target_layer: gold

model:
  fields:
    - name: region
      type: string
      required: true
    - name: total_revenue
      type: float
    - name: order_count
      type: integer

external_logic:
  type: python
  path: "transforms/revenue_summary.py"
  entrypoint: run
  args:
    fiscal_year: 2026
    include_refunds: false

quality:
  row_rules:
    - name: positive_revenue
      sql: "total_revenue >= 0"
"""

ext_contract_path = s.write_contract(ext_yaml, f"{EXT_DIR}/gold_revenue_summary.yaml")

# ── Step 3: Generate realistic source data ────────────────────────
import random

random.seed(42)

regions = ["EMEA", "APAC", "Americas", "LATAM"]
statuses = ["completed", "completed", "completed", "refunded", "pending"]

source_data = pl.DataFrame(
    {
        "order_id": list(range(1, 201)),
        "region": [random.choice(regions) for _ in range(200)],
        "amount": [round(random.uniform(10, 500), 2) for _ in range(200)],
        "status": [random.choice(statuses) for _ in range(200)],
    }
)

print("\n1. SOURCE DATA (200 orders across 4 regions):\n")
display(s.preview(source_data, 5))
print(f"   ... {s.row_count(source_data)} total rows")

# ── Step 4: Run the pipeline — LakeLogic calls your script ───────
proc = ll.DataProcessor(ext_contract_path, engine=ENGINE)
result = proc.run(source_data)

print("\n2. AFTER EXTERNAL LOGIC (your script aggregated by region):\n")
display(s.preview(result.good))

print("\n" + "─" * 60)
print("What happened:")
print("  1. LakeLogic validated 200 rows against the contract schema")
print("  2. Called transforms/revenue_summary.py → run(df, fiscal_year=2026)")
print("  3. Your script filtered refunds + aggregated by region")
print("  4. LakeLogic applied quality rules (positive_revenue >= 0)")
print("  5. Lineage metadata injected automatically")
print("\n✅ Your custom logic. LakeLogic's quality rules + lineage still apply.")

## What You Just Did

Six features that normally require six different engineering projects:

- ✅ **Engine portability** — same contract, Polars + DuckDB + Spark, identical results
- ✅ **Dimensional modeling** — SCD2 / merge / overwrite, declarative
- ✅ **Incremental processing** — `pipeline_log` watermark, no full rescans
- ✅ **Parallel execution** — concurrent multi-contract runs
- ✅ **Targeted backfills** — reprocess just the partitions that need it
- ✅ **Custom Python hooks** — drop in your own logic where YAML can't reach

Vendor lock-in: **zero**.

---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.

## 🧊 Multi-Format Materialization: Apache Iceberg

LakeLogic is table-format agnostic. While Delta Lake is the default, **Apache Iceberg** is rapidly becoming the open standard for analytical tables (especially across Snowflake, AWS, and Databricks workflows). You can natively output validated data directly into Iceberg format using either DuckDB or Spark simply by changing the `format` flag in your contract materialization block.

---
## Go Deeper — Explore by Capability

Each notebook below is **self-contained** and maps to one pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities). Pick the one that matters to you most.

| # | Notebook | What You'll See |
|---|---|---|
| 🚀 | **[Quickstart](00_quickstart.ipynb)** | One contract, every row accounted for, PII masked — in 5 minutes |
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

---

**Like what you saw?** ⭐ [Star us on GitHub](https://github.com/LakeLogic/LakeLogic) — it's how we know this matters.